# CS383: Data Science and Machine Learning
## In-Class Exercise (following Lecture 3) — Joins with SQL and Pandas

*Dr. Thitima Srivatanakul*

### Guiding question
**What happens when the answer to a question needs data from more than one table?**

### Learning objectives
By the end of this exercise, you should be able to:

- explain why real databases usually split related information across multiple tables instead of one big table;
- write an `INNER JOIN` and a `LEFT JOIN` in SQL;
- perform the equivalent operation in Pandas with `.merge()`;
- explain, with a concrete example, how `INNER JOIN` and `LEFT JOIN` can give different results;
- recognize why a join can produce missing (`NULL` / `NaN`) values, and what that means.

---

### Where this fits
This picks up exactly where Lecture 3 left off — same NYC 311 dataset, same idea of a SQLite database. Lecture 3 kept everything in one table. Today we deliberately split that table's context across a couple of small additional tables (the way real databases are usually organized) and learn to bring them back together with a **join**. Reshaping, real datetime handling, and messy real-world cleaning are still ahead — this exercise is joins, and only joins.

---

## Part 1 — Why Split Data Into Multiple Tables?

Imagine the `complaints` table from Lecture 3 also stored, for every single row, the full name of the agency that handled it: "New York City Police Department," "Department of Housing Preservation and Development," and so on, repeated for every one of thousands of matching rows.

That's wasteful, and worse, it's risky: if the agency's name is ever misspelled or updated in one row and not another, your data is now inconsistent. The standard fix is to store a short **code** in the big table, and keep the full names in a small separate **lookup table** — then join the two only when you need both pieces together.

### Setting up: complaints, plus two small lookup tables

We'll pull NYC 311 data again, this time including the `agency` code column, and add two small reference tables: one mapping agency codes to full names, and one mapping boroughs to population. Notice the agency lookup table is deliberately **incomplete** — a few codes that show up in `complaints` won't have a matching row. That's on purpose; we'll use it in Part 3.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import requests

SOCRATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    response = requests.get(
        SOCRATA_URL,
        params={
            "$limit": 8000,
            "$order": "created_date DESC",
            "$select": "unique_key,complaint_type,borough,agency,created_date",
        },
        timeout=8,
    )
    response.raise_for_status()
    complaints_df = pd.DataFrame(response.json())
    live = True
except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    # Agencies -- note "DSNY" and "DOB" appear here but will NOT appear in the lookup table below.
    agency_codes = ["NYPD", "HPD", "DOT", "DEP", "DSNY", "DOB"]
    complaints_df = pd.DataFrame({
        "unique_key": np.arange(1, n + 1),
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "agency": rng.choice(agency_codes, size=n, p=[0.30, 0.20, 0.15, 0.15, 0.12, 0.08]),
        "created_date": pd.date_range("2026-01-01", periods=n, freq="min").astype(str),
    })
    live = False

print(f"{'Live' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")

# A small, deliberately incomplete agency lookup table.
agencies_df = pd.DataFrame({
    "agency_code": ["NYPD", "HPD", "DOT", "DEP"],
    "agency_name": [
        "New York City Police Department",
        "Department of Housing Preservation and Development",
        "Department of Transportation",
        "Department of Environmental Protection",
    ],
})

# A small borough population lookup table (approximate, for teaching purposes).
boroughs_df = pd.DataFrame({
    "borough": ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"],
    "population": [1_630_000, 2_590_000, 2_270_000, 1_360_000, 490_000],
})

conn = sqlite3.connect("nyc311_lab.db")
complaints_df.to_sql("complaints", conn, if_exists="replace", index=False)
agencies_df.to_sql("agencies", conn, if_exists="replace", index=False)
boroughs_df.to_sql("boroughs", conn, if_exists="replace", index=False)

print("Tables ready: complaints, agencies, boroughs")

Live data: 8,000 311 records
Tables ready: complaints, agencies, boroughs


In [9]:
sorted(complaints_df["agency"].unique())

['DCWP',
 'DEP',
 'DHS',
 'DOB',
 'DOHMH',
 'DOT',
 'DPR',
 'DSNY',
 'EDC',
 'HPD',
 'NYPD',
 'OOS',
 'OTI',
 'TLC']

---

## Part 2 — INNER JOIN

An `INNER JOIN` combines rows from two tables where the join condition matches **in both tables**. If a row's `agency` code has no matching row in `agencies`, it's simply left out.

In [10]:
sql = """
SELECT complaints.unique_key, complaints.complaint_type, complaints.agency, agencies.agency_name
FROM complaints
JOIN agencies ON complaints.agency = agencies.agency_code
LIMIT 5;
"""
pd.read_sql_query(sql, conn)

,unique_key,complaint_type,agency,agency_name
0,70027941,Noise - Residential,NYPD,New York City Police Department
1,70026445,Noise - Vehicle,NYPD,New York City Police Department
2,70032610,Noise - Street/Sidewalk,NYPD,New York City Police Department
3,70033308,Illegal Parking,NYPD,New York City Police Department
4,70037864,Illegal Parking,NYPD,New York City Police Department


In [11]:
complaints_df.merge(
    agencies_df, left_on="agency", right_on="agency_code"
)[["unique_key", "complaint_type", "agency", "agency_name"]].head()

,unique_key,complaint_type,agency,agency_name
0,70027941,Noise - Residential,NYPD,New York City Police Department
1,70026445,Noise - Vehicle,NYPD,New York City Police Department
2,70032610,Noise - Street/Sidewalk,NYPD,New York City Police Department
3,70033308,Illegal Parking,NYPD,New York City Police Department
4,70037864,Illegal Parking,NYPD,New York City Police Department


`JOIN table2 ON t1.col = t2.col` in SQL maps to `df1.merge(df2, left_on="col1", right_on="col2")` in Pandas — `.merge()` is Pandas' join operation. Plain `JOIN` in SQL means `INNER JOIN` by default.

### How many rows disappeared?

In [15]:
inner_result = complaints_df.merge(agencies_df, left_on="agency", right_on="agency_code")

print(f"Rows in complaints:      {len(complaints_df):,}")
print(f"Rows after INNER JOIN:   {len(inner_result):,}")
print(f"Rows dropped:            {len(complaints_df) - len(inner_result):,}")

Rows in complaints:      8,000
Rows after INNER JOIN:   5,948
Rows dropped:            2,052


Those dropped rows are the complaints whose `agency` code (like `DSNY` or `DOB`) has no match in our incomplete `agencies` lookup table. An `INNER JOIN` silently drops them — which is exactly why you always want to check your row counts before and after a join.

---

## Part 3 — LEFT JOIN and Missing Matches

A `LEFT JOIN` keeps **every row from the left (first) table**, even if there's no match in the right table. Where there's no match, the right table's columns come back as `NULL` (SQL) or `NaN` (Pandas).

In [16]:
sql = """
SELECT complaints.unique_key, complaints.complaint_type, complaints.agency, agencies.agency_name
FROM complaints
LEFT JOIN agencies ON complaints.agency = agencies.agency_code
WHERE agencies.agency_name IS NULL
LIMIT 5;
"""
pd.read_sql_query(sql, conn)

,unique_key,complaint_type,agency,agency_name
0,70034615,Homeless Person Assistance,DHS,None
1,70036212,Rodent,DOHMH,None
2,70033623,Dirty Condition,DSNY,None
3,70038048,Litter Basket Request,DSNY,None
4,70028367,Consumer Complaint,DCWP,None


In [17]:
left_result = complaints_df.merge(agencies_df, left_on="agency", right_on="agency_code", how="left")
left_result[left_result["agency_name"].isna()][["unique_key", "complaint_type", "agency", "agency_name"]].head()

,unique_key,complaint_type,agency,agency_name
18,70034615,Homeless Person Assistance,DHS,NaN
35,70036212,Rodent,DOHMH,NaN
57,70033623,Dirty Condition,DSNY,NaN
64,70038048,Litter Basket Request,DSNY,NaN
66,70028367,Consumer Complaint,DCWP,NaN


`LEFT JOIN` in SQL maps to `how="left"` in `.merge()` (Pandas defaults to an inner join if you don't specify `how`). `IS NULL` in SQL is the equivalent of Pandas' `.isna()`. Notice the row count check from Part 2: a `LEFT JOIN` here would keep all 8,000-ish rows, agency name and all, with `NaN` standing in for the agencies our lookup table didn't cover.

---

## Part 4 — A Second Join: Complaints Per Capita by Borough

Raw complaint counts by borough mostly just reflect population — Brooklyn has more complaints partly because it has more people. Joining against the `boroughs` population table lets us ask a more honest question: which borough files the *most complaints relative to its size*?

In [22]:
sql = """
SELECT c.borough, COUNT(*) AS complaint_count, b.population,
       ROUND(1000.0 * COUNT(*) / b.population, 2) AS complaints_per_1000
FROM complaints AS c
JOIN boroughs AS b ON c.borough = b.borough
GROUP BY c.borough
ORDER BY complaints_per_1000 DESC;
"""
pd.read_sql_query(sql, conn)

,borough,complaint_count,population,complaints_per_1000
0,BRONX,1495,1360000,1.10
1,BROOKLYN,2657,2590000,1.03
2,QUEENS,2071,2270000,0.91
3,MANHATTAN,1461,1630000,0.90
4,STATEN ISLAND,313,490000,0.64


In [23]:
counts = complaints_df.groupby("borough").size().reset_index(name="complaint_count")
merged = counts.merge(boroughs_df, on="borough")
merged["complaints_per_1000"] = (1000 * merged["complaint_count"] / merged["population"]).round(2)
merged.sort_values("complaints_per_1000", ascending=False)

,borough,complaint_count,population,complaints_per_1000
0,BRONX,1495,1360000,1.10
1,BROOKLYN,2657,2590000,1.03
3,QUEENS,2071,2270000,0.91
2,MANHATTAN,1461,1630000,0.90
4,STATEN ISLAND,313,490000,0.64


Same pattern as Part 2 -- group and count first, then join the result against a lookup table -- except this time the join happens *after* an aggregation instead of before one. Both orders are valid; which one you reach for usually depends on which is easier to read for the question you're asking.

---

## Part 5 — Your Turn

**Question:** which complaint types most often come from an agency that's missing from our lookup table? Fill in the blanks to find out, using a `LEFT JOIN` and a filter for missing agency names.

In [ ]:
sql = """
SELECT complaint_type, COUNT(*) AS n
FROM complaints
LEFT JOIN agencies ON complaints.agency = agencies.agency_code
WHERE agencies.agency_name IS __________
GROUP BY complaint_type
ORDER BY n DESC
LIMIT 5;
"""
pd.read_sql_query(sql, conn)

---

### A note: Pandas also has `.join()`

Besides `.merge()`, Pandas has a `.join()` method — but it works differently: `.join()` combines DataFrames based on their **index**, not on any column you specify.

In [ ]:
# .merge() -- combine on a column, explicitly.
complaints_df.merge(boroughs_df, on="borough").head(2)

In [ ]:
# .join() -- combine on the index instead, which means setting up matching indices first.
complaints_indexed = complaints_df.set_index("borough")
boroughs_indexed = boroughs_df.set_index("borough")
complaints_indexed.join(boroughs_indexed, rsuffix="_ref").head(2)

`.join()` is a nice shorthand when your DataFrames already have clean, matching indices set up — but it's easy to get confusing results if you reach for it out of habit instead of `.merge()`. When in doubt, `.merge()` is the safer, more explicit default: it says exactly which column you're combining on, right there in the code. That's why it's the only one used throughout the rest of this exercise.

---

## Part 6 — Join Cheat Sheet

| Task | SQL | Pandas |
|---|---|---|
| Join, keep only matches | `JOIN` / `INNER JOIN` | `.merge(how="inner")` (the default) |
| Join, keep all left rows | `LEFT JOIN` | `.merge(how="left")` |
| Specify the join columns | `ON t1.col = t2.col` | `left_on="col1", right_on="col2"` (or `on="col"` if the names match) |
| Check for a missing match | `IS NULL` | `.isna()` |
| Join after grouping | aggregate first, then `JOIN` the result | `.groupby(...)`, then `.merge()` the result |

---

## Key Terms

- **Join**: combining rows from two tables based on a matching column.
- **Lookup table**: a small table mapping codes/keys to fuller, human-readable information.
- **INNER JOIN**: keeps only rows with a match in both tables.
- **LEFT JOIN**: keeps every row from the first (left) table, filling in `NULL`/`NaN` where there's no match.
- **`.merge()`**: Pandas' join operation, with a `how` argument (`"inner"`, `"left"`, `"right"`, `"outer"`) controlling which rows survive.
- **`NULL` / `NaN`**: SQL's and Pandas' respective ways of representing "no value here."

---

## Exit Ticket

1. In your own words, what's the difference between an `INNER JOIN` and a `LEFT JOIN`?
2. Why did the row count change after the `INNER JOIN` in Part 2, but not after the `LEFT JOIN` in Part 3?
3. Why store agency names in a separate lookup table instead of repeating them in every row of `complaints`?
4. What question would you want to answer that requires joining two tables together?

**Your responses:**

1.  
2.  
3.  
4.  